# Retail Return-Risk: Predicting Product Returns and Quantifying the Business Value

**Author:** Leonardo Flores · [LinkedIn](https://www.linkedin.com/in/leonardo-floresg/) · [GitHub](https://github.com/Luthiax)  
**Data:** Kaggle — "Retail Store Product Sales Dataset" (15,000 rows × 11 columns)  
**Last run:** auto-stamped on execution

---

## Business Question

> *Which product attributes are most strongly associated with high return rates — and if a retailer could reduce returns by 10%, what is that worth in net present value over three years?*

Product returns cost retailers between **5% and 15% of total sales** in handling, restocking, markdowns,
and reverse logistics. This project takes an end-to-end analytics approach:

1. **Explore** the data to understand the distribution of returns and key drivers.
2. **Model** return risk using a mix of explainable and ensemble methods.
3. **Translate** the best model's predictive power into a dollar-denominated business case.

The goal is not to squeeze the last 0.5% of R² — it is to produce an **actionable, defensible recommendation** a category manager or COO could act on.

## 1 · Setup and Data Loading

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

from src.data_pipeline import RetailDataPipeline
from src.feature_engineering import RetailFeatureEngineer
from src.financial_sim import FinancialRiskSimulator

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")

# ── Load & engineer features ──────────────────────────────────────────────────
pipeline = RetailDataPipeline(file_path="RetailStoreProductSalesDataset.csv")
engineer = RetailFeatureEngineer()

df_raw  = pipeline.load_and_clean()
df      = engineer.construct_features(df_raw)

print(f"Dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Target: return_rate  |  mean={df['return_rate'].mean():.4f}  |  P95={df['return_rate'].quantile(0.95):.4f}")
df.head()

## 2 · Target Distribution

We look at `return_rate` expressed as a **percentage** (fraction × 100) to make the axis intuitive.
The mean return rate is ~6.3% and the 95th-percentile threshold is ~9.2% — products above that are
considered *high-risk* for this analysis.

In [ ]:
return_pct = df["return_rate"] * 100
mean_val   = return_pct.mean()
p95_val    = return_pct.quantile(0.95)

fig, ax = plt.subplots(figsize=(10, 4))
sns.histplot(return_pct, bins=40, color="#7a8b99", kde=True, ax=ax, edgecolor="white")
ax.axvline(mean_val, color="#f39c12", linestyle="--", linewidth=2, label=f"Mean: {mean_val:.2f}%")
ax.axvline(p95_val,  color="#e74c3c", linestyle="--", linewidth=2, label=f"P95 (high-risk): {p95_val:.2f}%")
ax.set_title("Distribution of Product Return Rates", fontweight="bold")
ax.set_xlabel("Return Rate (%)")
ax.set_ylabel("Number of Transactions")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0f}%"))
ax.legend()
plt.tight_layout()
plt.savefig("nb_01_return_distribution.png", dpi=150)
plt.show()
print(f"Mean: {mean_val:.2f}% | Median: {return_pct.median():.2f}% | P95: {p95_val:.2f}% | Max: {return_pct.max():.2f}%")

## 3 · Exploratory Data Analysis

### 3a · Pearson Correlation Matrix

A quick look at pairwise linear correlations. We focus on the `return_rate` column to identify
features that move together with returns.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
corr = df.corr(numeric_only=True)
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            linewidths=0.5, ax=ax, cbar_kws={"shrink": 0.8})
ax.set_title("Pearson Correlation Matrix — All Features", fontweight="bold")
plt.tight_layout()
plt.savefig("nb_02_correlation.png", dpi=150)
plt.show()

### 3b · Return Rate by Customer Sentiment Decile

We bin `customer_sentiment` into 10 equal-frequency groups (deciles).
If the model's assumption holds — that lower sentiment predicts higher returns —
we expect a clear negative gradient from decile 1 to decile 10.

> **Leakage note:** `customer_sentiment` is treated as a *pre-purchase* rating signal
> (e.g., a star rating visible at the point of purchase). If it were actually a
> *post-purchase* satisfaction score, it would be partially tautological. This assumption
> is stated here and in `data/README.md` so reviewers can challenge it.

In [ ]:
df_plot = df.copy()
df_plot["sentiment_decile"] = pd.qcut(df_plot["customer_sentiment"], q=10, labels=False) + 1
summary = (
    df_plot.groupby("sentiment_decile", observed=False)["return_rate"]
    .mean()
    .mul(100)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(summary["sentiment_decile"], summary["return_rate"],
        marker="o", linewidth=2.5, color="#e7543d", markersize=8)
ax.set_title("Average Return Rate by Customer Sentiment Decile\n(Decile 1 = Lowest Sentiment, 10 = Highest)",
             fontweight="bold")
ax.set_xlabel("Customer Sentiment Decile")
ax.set_ylabel("Average Return Rate (%)")
ax.set_xticks(range(1, 11))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.1f}%"))
plt.tight_layout()
plt.savefig("nb_03_sentiment_decile.png", dpi=150)
plt.show()

### 3c · Price Competitiveness vs Return Rate

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.scatter(df["price_competitiveness"], df["return_rate"] * 100,
           alpha=0.15, s=8, color="#2b5c8f")
# Trend line
z = np.polyfit(df["price_competitiveness"], df["return_rate"] * 100, 1)
p = np.poly1d(z)
x_line = np.linspace(df["price_competitiveness"].min(), df["price_competitiveness"].max(), 200)
ax.plot(x_line, p(x_line), color="#e74c3c", linewidth=2, label=f"Trend (slope={z[0]:.3f})")
ax.set_title("Price Competitiveness vs Return Rate\n(price_competitiveness = price / competitor_price)",
             fontweight="bold")
ax.set_xlabel("Price Competitiveness Ratio")
ax.set_ylabel("Return Rate (%)")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0f}%"))
ax.legend()
plt.tight_layout()
plt.savefig("nb_04_price_competitiveness.png", dpi=150)
plt.show()

## 4 · Model Benchmark

We compare five models on the same 80/20 holdout split and 5-fold cross-validation.

### Why Ridge as the final model?

| Model | Test R² | Test RMSE | 5-fold CV R² | Notes |
|---|---|---|---|---|
| Dummy (mean baseline) | ~0.00 | ~0.012 | ~0.00 | Sanity check |
| **Ridge (α=2.976)** | **0.694** | **0.0099** | see below | Explainable coefficients; preferred |
| Random Forest | ~0.70 | ~0.0097 | similar | Black-box, needs SHAP |
| XGBoost | ~0.70 | ~0.0097 | similar | Black-box |
| LightGBM | ~0.70 | ~0.0097 | similar | Black-box |

Ridge is within **~0.01 R²** of the ensemble methods. For a *business deployment* context,
the explainability of a linear model — where a category manager can read "discount raises
returns by β" directly off a coefficient plot — is more valuable than squeezing an extra
fraction-of-a-percent of predictive accuracy. The 0.01 R² trade is not worth losing
stakeholder trust in the model's decisions.

In [ ]:
# ── Train/test split & scaling ────────────────────────────────────────────────
X_train, X_test, y_train, y_test, X_test_raw = pipeline.split_and_scale(df)

models = {
    "Dummy (mean)":   DummyRegressor(strategy="mean"),
    "Ridge α=2.976":  Ridge(alpha=2.976, random_state=RANDOM_STATE),
    "Random Forest":  RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1),
}

# Optional: XGBoost and LightGBM (skip gracefully if not installed)
try:
    from xgboost import XGBRegressor
    models["XGBoost"] = XGBRegressor(n_estimators=200, learning_rate=0.05,
                                      random_state=RANDOM_STATE, verbosity=0)
except ImportError:
    print("XGBoost not installed — skipping")

try:
    from lightgbm import LGBMRegressor
    models["LightGBM"] = LGBMRegressor(n_estimators=200, learning_rate=0.05,
                                        random_state=RANDOM_STATE, verbose=-1)
except ImportError:
    print("LightGBM not installed — skipping")

results = []
fitted_models = {}
for name, m in models.items():
    m.fit(X_train, y_train)
    y_pred = m.predict(X_test)
    r2   = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    cv   = cross_val_score(m, X_train, y_train, cv=5, scoring="r2").mean()
    results.append({"Model": name, "Test R²": round(r2, 4), "Test RMSE": round(rmse, 5), "5-fold CV R²": round(cv, 4)})
    fitted_models[name] = m

pd.DataFrame(results).sort_values("Test R²", ascending=False).reset_index(drop=True)

## 5 · Ridge Regression Coefficients

The standardized coefficients show the direction and relative magnitude of each feature's
effect on return rate. Green bars reduce returns; red bars increase them.

In [ ]:
ridge = fitted_models["Ridge α=2.976"]
coef_df = pd.DataFrame({
    "Feature": X_train.columns,
    "Coefficient": ridge.coef_
}).sort_values("Coefficient")

colors = ["#5cb85c" if c < 0 else "#d9534f" for c in coef_df["Coefficient"]]
coef_df["_color"] = colors

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(x="Coefficient", y="Feature", data=coef_df,
            hue="Feature", palette=dict(zip(coef_df["Feature"], colors)), legend=False, ax=ax)
ax.set_title("Ridge Regression Coefficients\n(Green = reduces returns | Red = increases returns)",
             fontweight="bold")
ax.set_xlabel("Standardized Coefficient")
ax.set_ylabel("")
ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
plt.tight_layout()
plt.savefig("nb_05_coefficients.png", dpi=150)
plt.show()
print("Top 3 return-REDUCING features:", coef_df["Feature"].iloc[:3].tolist())
print("Top 3 return-INCREASING features:", coef_df["Feature"].iloc[-3:].tolist())

## 6 · NPV Business Case

### Financial assumptions

| Parameter | Value | Rationale |
|---|---|---|
| Total returns per year | 30,000 | Enterprise-scale retailer baseline |
| Project capex | $50,000 | Data infrastructure + model development |
| Annual opex | $15,000 | Model monitoring, retraining, compute |
| Discount rate | 10% | Mid-cap retailer WACC approximation |
| Horizon | 3 years | Typical analytics tooling depreciation cycle |

The grid below shows **3-Year NPV in $000s** for combinations of:
- **Handling cost per return** ($5, $10, $15, $20) — the variable cost of processing one returned unit
- **Relative reduction in return rate** (5%, 10%, 15%, 20%) — the model's operational impact

Green cells are profitable deployments; red cells don't pay back the investment.

In [ ]:
simulator = FinancialRiskSimulator(capex=50_000, opex_annual=15_000, discount_rate=0.10)
TOTAL_RETURNS = 30_000

cost_levels      = [5, 10, 15, 20]
reduction_levels = [0.05, 0.10, 0.15, 0.20]

grid = []
for cost in cost_levels:
    row = []
    for red in reduction_levels:
        npv_k = simulator.simulate_3year_npv(
            return_handling_cost=cost, target_reduction=red,
            total_returns_annual=TOTAL_RETURNS
        ) / 1000
        row.append(round(npv_k, 1))
    grid.append(row)

npv_array = np.array(grid)

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(npv_array, annot=True, fmt=".1f", cmap="RdYlGn", center=0,
            xticklabels=["5% Reduction","10% Reduction","15% Reduction","20% Reduction"],
            yticklabels=["$5/return","$10/return","$15/return","$20/return"],
            cbar_kws={"label": "3-Year NPV ($000s)"}, linewidths=0.5, ax=ax)
ax.set_title("Business ROI: 3-Year NPV Sensitivity Grid ($000s)\n"
             "Green = profitable deployment | Red = investment not recovered",
             fontweight="bold")
ax.set_xlabel("Relative Reduction in Return Rate (Model Impact)")
ax.set_ylabel("Avg. Cost per Return (Handling + Logistics)")
plt.tight_layout()
plt.savefig("nb_06_npv_grid.png", dpi=150)
plt.show()

base_npv = simulator.simulate_3year_npv(return_handling_cost=20.0, target_reduction=0.10,
                                         total_returns_annual=TOTAL_RETURNS)
print(f"Base-case NPV ($20/return, 10% reduction, 30k returns/yr): ${base_npv:,.2f}")

## 7 · Model Error by Price Tier

Where does the model struggle most? We segment the test set into three price tiers
(Low-Cost, Mid-Tier, Premium) and compare mean absolute error (MAE) per tier.
Understanding error segments helps prioritise retraining data collection.

In [ ]:
y_pred_ridge  = fitted_models["Ridge α=2.976"].predict(X_test)
diag_df       = X_test_raw.copy()
diag_df["absolute_error"] = abs(y_test.values - y_pred_ridge)
diag_df["price_tier"] = pd.qcut(diag_df["price"], q=3, labels=["Low-Cost", "Mid-Tier", "Premium"])

tier_mae = (
    diag_df.groupby("price_tier", observed=False)["absolute_error"]
    .agg(["mean", "median", "count"])
    .rename(columns={"mean": "Mean MAE", "median": "Median MAE", "count": "N"})
    .mul({"Mean MAE": 100, "Median MAE": 100, "N": 1})  # convert fraction error to %
    .round(3)
)

fig, ax = plt.subplots(figsize=(8, 4))
colors_tier = ["#5bc0de", "#5cb85c", "#d9534f"]
bars = ax.bar(tier_mae.index, tier_mae["Mean MAE"], color=colors_tier, edgecolor="white", width=0.5)
ax.bar_label(bars, fmt="%.3f%%", padding=3)
ax.set_title("Mean Absolute Error by Price Tier\n(Lower is better)", fontweight="bold")
ax.set_ylabel("Mean |Actual − Predicted| (return-rate pp)")
ax.set_ylim(0, tier_mae["Mean MAE"].max() * 1.3)
plt.tight_layout()
plt.savefig("nb_07_error_by_tier.png", dpi=150)
plt.show()
print(tier_mae.to_string())

## 8 · Conclusions and Recommendations

### What we built

A Ridge regression model (R² = 0.694, RMSE = 0.0099 on a 0–1 return-rate scale) that
predicts product-level return risk from 14 features (10 raw + 4 engineered) derived from
pre-purchase signals. The model's predictions were then plugged into a 3-year NPV simulation
to quantify the financial upside of acting on the model's output.

---

### Three business recommendations

1. **Focus markdown/pricing on high-risk products first.**  
   Products above the P95 return threshold (>9.2%) account for a disproportionate share of
   return cost. The model can flag these before shipping promotions — the category team should
   review the top-5% flagged SKUs every week.

2. **Increase customer sentiment (product rating) as a lever.**  
   `customer_sentiment` is the strongest negative driver of returns. Investing in better
   product descriptions, accurate photos, and size guides is likely to raise pre-purchase
   sentiment and reduce post-purchase disappointment — and therefore returns.

3. **Use the breakeven analysis to size the business case.**  
   At a $20 handling cost per return (enterprise logistics), the model only needs to enable
   a **5.85% reduction in returns** to pay back the $50k capex + $15k/yr opex over 3 years.
   This is a conservative threshold — most analytics deployments achieve 10–15% reduction.
   At $10/return the bar rises to 11.7%, which the team should validate against their actual
   logistics cost before committing budget.

---

### Three honest limitations

| # | Limitation | Mitigation |
|---|---|---|
| 1 | **`customer_sentiment` leakage risk.** Interpreted here as pre-purchase rating; if it is actually post-purchase satisfaction, R² is artificially inflated. | Replace with lagged product review score from prior period in production. |
| 2 | **Random (not time-based) train/test split.** An 80/20 random split does not respect the temporal order of sales data. A time-based split (train on months 1–18, test on months 19–24) would give a more honest out-of-sample estimate. | Implement a time-based split before production deployment. |
| 3 | **Constant 30,000 annual returns assumed.** The NPV grid uses a fixed volume. In reality, return volume is seasonal (Q4 spikes ~40%). | Run the NPV simulation with seasonal return-volume curves for a more precise business case. |

---

### About

**Leonardo Flores** — Bilingual (EN/ES) Business Analytics professional, Lima, Peru.  
Specialization in Business Analytics and AI.

🔗 [LinkedIn](https://www.linkedin.com/in/leonardo-floresg/) · 🐙 [GitHub](https://github.com/Luthiax)